# Direct Marketing with Amazon SageMaker Autopilot
---

---

<div style="border: 2px solid #ff9900; border-radius: 8px; padding: 15px; background-color: #fff3e0; margin-bottom: 10px;">
<strong>⚠️ Compatibility Notice:</strong> This notebook has been tested using <strong>SageMaker Distribution Image 3.7.0</strong> and the <strong>SageMaker Python SDK version 3.4.0</strong>.
</div>

In [ ]:
!pip install -q -U "sagemaker==3.4.0"

In [ ]:
# Restart kernel to pick up the updated packages
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
import sagemaker
import sagemaker.core
import sagemaker.train
import sagemaker.serve
import sagemaker.mlops
from importlib.metadata import version

print(f"sagemaker: {version('sagemaker')}")
print(f"sagemaker.core: {version('sagemaker.core')}")
print(f"sagemaker.train: {version('sagemaker.train')}")
print(f"sagemaker.server: {version('sagemaker.serve')}")
print(f"sagemaker.mlops: {version('sagemaker.mlops')}")

## Contents

1. [Introduction](#Introduction)
1. [Prerequisites](#Prerequisites)
1. [Downloading the dataset](#Downloading)
1. [Upload the dataset to Amazon S3](#Uploading)
1. [Setting up the SageMaker Autopilot Job](#Settingup)
1. [Launching the SageMaker Autopilot Job](#Launching)
1. [Tracking Sagemaker Autopilot Job Progress](#Tracking)
1. [Results](#Results)
1. [Cleanup](#Cleanup)

## Introduction

Amazon SageMaker Autopilot is an automated machine learning (commonly referred to as AutoML) solution for tabular datasets. You can use SageMaker Autopilot in different ways: on autopilot (hence the name) or with human guidance, without code through SageMaker Studio, or using the AWS SDKs. This notebook, as a first glimpse, will use the AWS SDKs to simply create and deploy a machine learning model.

A typical introductory task in machine learning (the "Hello World" equivalent) is one that uses a dataset to predict whether a customer will enroll for a term deposit at a bank, after one or more phone calls. For more information about the task and the dataset used, see [Bank Marketing Data Set](https://archive.ics.uci.edu/ml/datasets/bank+marketing).

Direct marketing, through mail, email, phone, etc., is a common tactic to acquire customers.  Because resources and a customer's attention are limited, the goal is to only target the subset of prospects who are likely to engage with a specific offer.  Predicting those potential customers based on readily available information like demographics, past interactions, and environmental factors is a common machine learning problem. You can imagine that this task would readily translate to marketing lead prioritization in your own organization.

This notebook demonstrates how you can use Autopilot on this dataset to get the most accurate ML pipeline through exploring a number of potential options, or "candidates". Each candidate generated by Autopilot consists of two steps. The first step performs automated feature engineering on the dataset and the second step trains and tunes an algorithm to produce a model. When you deploy this model, it follows similar steps. Feature engineering followed by inference, to decide whether the lead is worth pursuing or not. The notebook contains instructions on how to train the model as well as to deploy the model to perform batch predictions on a set of leads. Where it is possible, use the Amazon SageMaker Python SDK, a high level SDK, to simplify the way you interact with Amazon SageMaker.

Other examples demonstrate how to customize models in various ways. For instance, models deployed to devices typically have memory constraints that need to be satisfied as well as accuracy. Other use cases have real-time deployment requirements and latency constraints. For now, keep it simple.

## Prerequisites

Before you start the tasks in this tutorial, do the following:

- The Amazon Simple Storage Service (Amazon S3) bucket and prefix that you want to use for training and model data. This should be within the same Region as Amazon SageMaker training. The code below will create, or if it exists, use, the default bucket.
- The IAM role to give Autopilot access to your data. See the Amazon SageMaker documentation for more information on IAM roles: https://docs.aws.amazon.com/sagemaker/latest/dg/security-iam.html

In [ ]:
# cell 01
from sagemaker.core.helper.session_helper import Session, get_execution_role
import boto3

session = Session()
bucket = session.default_bucket()
prefix = 'sagemaker/autopilot-dm'
role = get_execution_role()
region = session.boto_region_name

sm = boto3.client('sagemaker', region_name=region)

## Downloading the dataset<a name="Downloading"></a>
Download the [direct marketing dataset](!wget -N https://sagemaker-sample-data-us-west-2.s3-us-west-2.amazonaws.com/autopilot/direct_marketing/bank-additional.zip) from the sample data s3 bucket. 

\[Moro et al., 2014\] S. Moro, P. Cortez and P. Rita. A Data-Driven Approach to Predict the Success of Bank Telemarketing. Decision Support Systems, Elsevier, 62:22-31, June 2014

In [ ]:
# cell 02
local_data_path = './bank-additional/bank-additional-full.csv'


## Upload the dataset to Amazon S3<a name="Uploading"></a>

Before you run Autopilot on the dataset, first perform a check of the dataset to make sure that it has no obvious errors. The Autopilot process can take long time, and it's generally a good practice to inspect the dataset before you start a job. This particular dataset is small, so you can inspect it in the notebook instance itself. If you have a larger dataset that will not fit in a notebook instance memory, inspect the dataset offline using a big data analytics tool like Apache Spark. [Deequ](https://github.com/awslabs/deequ) is a library built on top of Apache Spark that can be helpful for performing checks on large datasets. Autopilot is capable of handling datasets up to 5 GB.


Read the data into a Pandas data frame and take a look.

In [ ]:
# cell 03
import pandas as pd

data = pd.read_csv(local_data_path)
pd.set_option('display.max_columns', 500)     # Make sure we can see all of the columns
pd.set_option('display.max_rows', 10)         # Keep the output on one page
data

Note that there are 20 features to help predict the target column 'y'.

Amazon SageMaker Autopilot takes care of preprocessing your data for you. You do not need to perform conventional data preprocssing techniques such as handling missing values, converting categorical features to numeric features, scaling data, and handling more complicated data types.

Moreover, splitting the dataset into training and validation splits is not necessary. Autopilot takes care of this for you. You may, however, want to split out a test set. That's next, although you use it for batch inference at the end instead of testing the model.


### Reserve some data for calling batch inference on the model

Divide the data into training and testing splits. The training split is used by SageMaker Autopilot. The testing split is reserved to perform inference using the suggested model.


In [ ]:
# cell 04
train_data = data.sample(frac=0.8,random_state=200)

test_data = data.drop(train_data.index)

test_data_no_target = test_data.drop(columns=['y'])

### Upload the dataset to Amazon S3
Copy the file to Amazon Simple Storage Service (Amazon S3) in a .csv format for Amazon SageMaker training to use.

In [ ]:
# cell 05
train_file = 'train_data.csv';
train_data.to_csv(train_file, index=False, header=True)
train_data_s3_path = session.upload_data(path=train_file, key_prefix=prefix + "/train")
print('Train data uploaded to: ' + train_data_s3_path)

test_file = 'test_data.csv';
test_data_no_target.to_csv(test_file, index=False, header=False)
test_data_s3_path = session.upload_data(path=test_file, key_prefix=prefix + "/test")
print('Test data uploaded to: ' + test_data_s3_path)

## Setting up the SageMaker Autopilot Job<a name="Settingup"></a>

After uploading the dataset to Amazon S3, you can invoke Autopilot to find the best ML pipeline to train a model on this dataset. 

The required inputs for invoking a Autopilot job are:
* Amazon S3 location for input dataset and for all output artifacts
* Name of the column of the dataset you want to predict (`y` in this case) 
* An IAM role

Currently Autopilot supports only tabular datasets in CSV format. Either all files should have a header row, or the first file of the dataset, when sorted in alphabetical/lexical order, is expected to have a header row.

In [ ]:
# cell 06
from sagemaker.core.resources import AutoMLJobV2
from sagemaker.core.shapes import (
    AutoMLJobChannel, AutoMLDataSource, AutoMLS3DataSource,
    AutoMLOutputDataConfig, AutoMLProblemTypeConfig, TabularJobConfig,
    AutoMLJobCompletionCriteria, AutoMLJobObjective,
)

auto_ml_job_input_data_config = [
    AutoMLJobChannel(
        data_source=AutoMLDataSource(
            s3_data_source=AutoMLS3DataSource(
                s3_data_type="S3Prefix",
                s3_uri=f's3://{bucket}/{prefix}/train',
            )
        )
    )
]

output_data_config = AutoMLOutputDataConfig(
    s3_output_path=f's3://{bucket}/{prefix}/output'
)

auto_ml_problem_type_config = AutoMLProblemTypeConfig(
    tabular_job_config=TabularJobConfig(
        completion_criteria=AutoMLJobCompletionCriteria(max_candidates=5),
        problem_type="BinaryClassification",
        target_attribute_name="y",
    )
)

auto_ml_job_objective = AutoMLJobObjective(metric_name="Accuracy")

You can also specify the type of problem you want to solve with your dataset (`Regression, MulticlassClassification, BinaryClassification`). In case you are not sure, SageMaker Autopilot will infer the problem type based on statistics of the target column (the column you want to predict). 

You have the option to limit the running time of a SageMaker Autopilot job by providing either the maximum number of pipeline evaluations or candidates (one pipeline evaluation is called a `Candidate` because it generates a candidate model) or providing the total time allocated for the overall Autopilot job. Under default settings, this job takes about four hours to run. This varies between runs because of the nature of the exploratory process Autopilot uses to find optimal training parameters.

## Launching the SageMaker Autopilot Job<a name="Launching"></a>

You can now launch the Autopilot job by calling the `create_auto_ml_job_v2` API. https://docs.aws.amazon.com/cli/latest/reference/sagemaker/create-auto-ml-job-v2.html

In [ ]:
# cell 07
from time import gmtime, strftime, sleep
timestamp_suffix = strftime('%d-%H-%M-%S', gmtime())

auto_ml_job_name = 'automl-banking-' + timestamp_suffix
print('AutoMLJobName: ' + auto_ml_job_name)

automl_job = AutoMLJobV2.create(
    auto_ml_job_name=auto_ml_job_name,
    auto_ml_job_input_data_config=auto_ml_job_input_data_config,
    output_data_config=output_data_config,
    auto_ml_job_objective=auto_ml_job_objective,
    auto_ml_problem_type_config=auto_ml_problem_type_config,
    role_arn=role,
)
print(f"Job created: {automl_job.auto_ml_job_arn}")

## Tracking SageMaker Autopilot job progress<a name="Tracking"></a>
SageMaker Autopilot job consists of the following high-level steps : 
* Analyzing Data, where the dataset is analyzed and Autopilot comes up with a list of ML pipelines that should be tried out on the dataset. The dataset is also split into train and validation sets.
* Feature Engineering, where Autopilot performs feature transformation on individual features of the dataset as well as at an aggregate level.
* Model Tuning, where the top performing pipeline is selected along with the optimal hyperparameters for the training algorithm (the last stage of the pipeline). 

In [ ]:
# cell 08
automl_job.wait(poll=30)
automl_job.refresh()
print(f"Status: {automl_job.auto_ml_job_status} - {automl_job.auto_ml_job_secondary_status}")

## Results

Now use the describe_auto_ml_job_v2 API to look up the best candidate selected by the SageMaker Autopilot job. 

In [ ]:
# cell 09
best_candidate = automl_job.best_candidate
best_candidate_name = best_candidate.candidate_name
print(f"CandidateName: {best_candidate_name}")
print(f"MetricName: {best_candidate.final_auto_ml_job_objective_metric.metric_name}")
print(f"MetricValue: {best_candidate.final_auto_ml_job_objective_metric.value}")

### Perform batch inference using the best candidate

Now that you have successfully completed the SageMaker Autopilot job on the dataset, create a model from any of the candidates by using [Inference Pipelines](https://docs.aws.amazon.com/sagemaker/latest/dg/inference-pipelines.html). 

In [ ]:
# cell 10
from sagemaker.core.resources import Model
from sagemaker.core.shapes import ContainerDefinition

model_name = 'automl-banking-model-' + timestamp_suffix

# Convert candidate inference containers to ContainerDefinition objects
containers = [
    ContainerDefinition(
        image=c.image,
        model_data_url=c.model_data_url,
        environment=c.environment,
    )
    for c in best_candidate.inference_containers
]

model = Model.create(
    model_name=model_name,
    containers=containers,
    execution_role_arn=role,
)
print(f"Model: {model.model_name}")

You can use batch inference by using Amazon SageMaker batch transform. The same model can also be deployed to perform online inference using Amazon SageMaker hosting.

In [ ]:
# cell 11
from sagemaker.core.resources import TransformJob
from sagemaker.core.shapes import (
    TransformInput, TransformOutput, TransformResources,
    TransformDataSource, TransformS3DataSource,
)

transform_job_name = 'automl-banking-transform-' + timestamp_suffix

transform_job = TransformJob.create(
    transform_job_name=transform_job_name,
    model_name=model_name,
    transform_input=TransformInput(
        data_source=TransformDataSource(
            s3_data_source=TransformS3DataSource(
                s3_data_type="S3Prefix",
                s3_uri=test_data_s3_path,
            )
        ),
        content_type="text/csv",
        compression_type="None",
        split_type="Line",
    ),
    transform_output=TransformOutput(
        s3_output_path=f's3://{bucket}/{prefix}/inference-results',
    ),
    transform_resources=TransformResources(
        instance_type="ml.m5.4xlarge",
        instance_count=1,
    ),
)
print(f"Transform job created: {transform_job_name}")

Watch the transform job for completion.

In [ ]:
# cell 12
transform_job.wait(poll=30)
transform_job.refresh()
print(f"Status: {transform_job.transform_job_status}")

Now let's view the results of the transform job:

In [ ]:
# cell 13
s3_output_key = '{}/inference-results/test_data.csv.out'.format(prefix);
local_inference_results_path = 'inference_results.csv'

s3 = boto3.resource('s3')
inference_results_bucket = s3.Bucket(session.default_bucket())

inference_results_bucket.download_file(s3_output_key, local_inference_results_path);

data = pd.read_csv(local_inference_results_path, sep=';')
pd.set_option('display.max_rows', 10)         # Keep the output on one page
data

### View other candidates explored by SageMaker Autopilot
You can view all the candidates (pipeline evaluations with different hyperparameter combinations) that were explored by SageMaker Autopilot and sort them by their final performance metric.

In [ ]:
# cell 14
candidates = sm.list_candidates_for_auto_ml_job(
    AutoMLJobName=auto_ml_job_name, SortBy='FinalObjectiveMetricValue'
)['Candidates']
for i, candidate in enumerate(candidates, 1):
    print(f"{i}  {candidate['CandidateName']}  {candidate['FinalAutoMLJobObjectiveMetric']['Value']}")

### Candidate Generation Notebook
    
Sagemaker AutoPilot also auto-generates a Candidate Definitions notebook. This notebook can be used to interactively step through the various steps taken by the Sagemaker Autopilot to arrive at the best candidate. This notebook can also be used to override various runtime parameters like parallelism, hardware used, algorithms explored, feature extraction scripts and more.
    
The notebook can be downloaded from the following Amazon S3 location:

In [ ]:
# cell 15
print(automl_job.auto_ml_job_artifacts.candidate_definition_notebook_location)

### Data Exploration Notebook
Sagemaker Autopilot also auto-generates a Data Exploration notebook, which can be downloaded from the following Amazon S3 location:

In [ ]:
# cell 16
print(automl_job.auto_ml_job_artifacts.data_exploration_notebook_location)

## Cleanup

The Autopilot job creates many underlying artifacts such as dataset splits, preprocessing scripts, or preprocessed data, etc. This code, when un-commented, deletes them. This operation deletes all the generated models and the auto-generated notebooks as well. 

In [ ]:
# cell 17
#s3 = boto3.resource('s3')
#bucket = s3.Bucket(bucket)

#job_outputs_prefix = '{}/output/{}'.format(prefix,auto_ml_job_name)
#bucket.objects.filter(Prefix=job_outputs_prefix).delete()